<a id="top"></a>

# Demo: carry the eval set across, and find the trace in Langfuse

```yaml
title: "Demo: carry the eval set across, and find the trace in Langfuse"
keywords: langgraph, eval, eval set, carry forward, adapter, runresult, common.evals, tool_trajectory, pass rate, langfuse, tracing, generation span, behavioral equivalence, teacher demo
estimated duration: 14
```

> **Lesson:** L14. Teacher demo -- Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L14/demos_or_activities.md).
> The **eval** runs offline (a deterministic `StubChat`) so the pass table is reproducible -- the
> lesson is the *adapter + harness transferring from L12*, not the model. The **Langfuse** beat runs
> live when `ANTHROPIC_API_KEY` + Langfuse are configured, and falls back to reading the trace
> inline otherwise. Clear any live output before committing.

## Contents

- [1. Setup -- the agent, the harness, the tracer](#1-setup----the-agent-the-harness-the-tracer)
- [2. The adapter: graph output to RunResult](#2-the-adapter-graph-output-to-runresult)
- [3. Carry the L12 eval set across](#3-carry-the-l09-eval-set-across)
- [4. Find the trace in the same Langfuse](#4-find-the-trace-in-the-same-langfuse)
- [5. Takeaways](#5-takeaways)

## 1. Setup -- the agent, the harness, the tracer

Everything reused: the shared tools, the offline `StubChat` (scripted for the two L10/L11 tasks),
the `build_agent(chat)` helper, **and the L12 eval harness imported straight from
[`common/evals.py`](../../common/evals.py)** -- the same `EvalCase` / `evaluate` / `tool_trajectory`
you used on the hand-rolled loop. That import *is* "carry it forward."

In [1]:
from typing import Annotated, Any, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import StructuredTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common import tools as T
from fluffy_potato_curriculum.common.agent_loop import RunResult
from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_anthropic_key,
)
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    evaluate,
    tool_trajectory,
)
from fluffy_potato_curriculum.common.tracing import TraceEvent

SONNET = "claude-sonnet-4-6"
calculator = StructuredTool.from_function(T.calculator, name="calculator", description="arithmetic")
lookup = StructuredTool.from_function(T.lookup, name="lookup", description="city population")
flaky_fetch = StructuredTool.from_function(
    T.flaky_fetch, name="flaky_fetch", description="fetch url"
)
LC_TOOLS = [calculator, lookup, flaky_fetch]

CHAIN_TASK = "Use the calculator to compute 21 * 2, then tell me the population of Tokyo."
CRASH_TASK = "Fetch the URL https://crash and tell me what happened."
LIVE = bool(get_settings().anthropic_api_key)


class StubChat:
    """Offline, scripted for BOTH canonical tasks so the eval is deterministic and keyless."""

    def bind_tools(self, tools: list[object]) -> "StubChat":
        return self

    def invoke(self, messages: list[BaseMessage]) -> AIMessage:
        human = next((m for m in messages if isinstance(m, HumanMessage)), None)
        task = str(human.content).lower() if human else ""
        ai_turns = sum(1 for m in messages if isinstance(m, AIMessage))
        if "crash" in task or "fetch" in task:
            if ai_turns == 0:
                return AIMessage(
                    content="",
                    tool_calls=[
                        {"name": "flaky_fetch", "args": {"url": "https://crash"}, "id": "f1"}
                    ],
                )
            return AIMessage(content="The fetch failed (connection reset); I stopped.")
        if ai_turns == 0:
            return AIMessage(
                content="",
                tool_calls=[{"name": "calculator", "args": {"expression": "21*2"}, "id": "c1"}],
            )
        if ai_turns == 1:
            return AIMessage(
                content="", tool_calls=[{"name": "lookup", "args": {"city": "Tokyo"}, "id": "l1"}]
            )
        return AIMessage(content="21*2 = 42, and Tokyo's population is 37000000.")


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    step: int


def build_agent(chat: Any) -> Any:
    """The same shallow agent, over whatever chat client is passed (stub or ChatAnthropic)."""

    def agent_node(state: AgentState) -> dict[str, object]:
        return {
            "messages": [chat.bind_tools(LC_TOOLS).invoke(state["messages"])],
            "step": state["step"] + 1,
        }

    def route(state: AgentState) -> str:
        last = state["messages"][-1]
        return "tools" if isinstance(last, AIMessage) and last.tool_calls else END

    b = StateGraph(AgentState)
    b.add_node("agent", agent_node)
    b.add_node("tools", ToolNode(LC_TOOLS, handle_tool_errors=True))
    b.set_entry_point("agent")
    b.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    b.add_edge("tools", "agent")
    return b.compile()


eval_app = build_agent(StubChat())  # the offline agent the eval runs against

[↑ Back to top](#top)

## 2. The adapter: graph output to RunResult

L12's scorers read a `RunResult` (`final_text`, `trace`). The graph returns a **state dict** (a
`messages` list). One thin **adapter** maps the second into the first, so the *same* `EvalCase`s and
scorers run **unchanged** against the rebuild. This adapter is the entire mechanism behind "bring
your eval set across."

In [2]:
def to_run_result(final_state: dict[str, Any]) -> RunResult:
    """Map the graph's output into the shape common/evals.py scores.

    Each tool call in the message history becomes a `tool` span (so `tool_trajectory` reads the
    path); the last assistant text becomes `final_text`.
    """
    msgs: list[BaseMessage] = final_state["messages"]
    trace: list[TraceEvent] = []
    final_text = ""
    for m in msgs:
        if isinstance(m, AIMessage):
            for call in m.tool_calls:
                trace.append(
                    TraceEvent(
                        run_id=call["id"],
                        trace_id="graph",
                        run_type="tool",
                        name=call["name"],
                        inputs=dict(call["args"]),
                    )
                )
            if not m.tool_calls:
                final_text = m.text
    iterations = sum(1 for m in msgs if isinstance(m, AIMessage))
    return RunResult(
        final_text=final_text, iterations=iterations, termination="natural", trace=trace
    )


# Sanity check the adapter on one run:
demo_state = eval_app.invoke({"messages": [HumanMessage(CHAIN_TASK)], "step": 0})
demo_run = to_run_result(demo_state)
print("final_text:", demo_run.final_text)
print("trajectory:", tool_trajectory(demo_run))  # read off the adapted trace, exactly like L12

final_text: 21*2 = 42, and Tokyo's population is 37000000.
trajectory: ['calculator', 'lookup']


[↑ Back to top](#top)

## 3. Carry the L12 eval set across

Now run an eval set against the LangGraph rebuild. These cases are the L12 style -- an input task
plus what "good" means (an expected answer and/or an expected **tool trajectory**). The scorers are
the L12 scorers; the only new piece is `run_case`, which invokes the graph and adapts its output.

The payoff: **the same cases that passed on the L10 loop pass on the graph.** Same cases, different
implementation, nothing regressed -- the cleanest proof the rebuild is correct.

In [3]:
def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    want = example.reference_outputs.get("answer")
    ok = True if want is None else (want in run.final_text)
    return EvalResult(key="answer_correct", score=ok)


def trajectory_ok(*, run: RunResult, example: EvalCase) -> EvalResult:
    want = example.reference_outputs["expected_tools"]
    return EvalResult(key="trajectory_ok", score=tool_trajectory(run) == want)


cases = [
    EvalCase(
        id="chain_calc_lookup",
        inputs={"task": CHAIN_TASK},
        reference_outputs={"answer": "37000000", "expected_tools": ["calculator", "lookup"]},
    ),
    EvalCase(
        id="flaky_crash_recover",
        inputs={"task": CRASH_TASK},
        reference_outputs={"expected_tools": ["flaky_fetch"]},
    ),
]


def run_case(case: EvalCase) -> RunResult:
    """The L12 'target': produce a RunResult for one case -- here by invoking the GRAPH."""
    state = eval_app.invoke({"messages": [HumanMessage(case.inputs["task"])], "step": 0})
    return to_run_result(state)


report = evaluate(run_case, cases, [answer_correct, trajectory_ok])
print(report.table())

case                 answer_correct  trajectory_ok  ALL
chain_calc_lookup    1/1             1/1            1/1
flaky_crash_recover  1/1             1/1            1/1


[↑ Back to top](#top)

## 4. Find the trace in the same Langfuse

Tracing transfers too. Attach the **Langfuse callback** (the same self-hosted instance from L11) and
the graph's run lands in the **same dashboard**, rendered as the **same spans** you already know -- a
GENERATION per model call, a SPAN per tool call -- next to your hand-rolled L11 traces. Only the
trace's *packaging* changed.

Keyless, you can still read the run **inline** -- the message list *is* the trace, the same
model-call / tool-call / state-update story L11 taught you to read.

In [4]:
# Read the run inline -- the in-memory trace, the same story L11 taught you to read:
print("inline trace (graph run):")
for m in demo_state["messages"]:
    asked = [c["name"] for c in m.tool_calls] if isinstance(m, AIMessage) else []
    print(f"  {type(m).__name__:12} {('-> ' + ', '.join(asked)) if asked else str(m.content)[:48]}")

# Live: send the SAME run to the L11 Langfuse dashboard as GENERATION/SPAN spans.
if LIVE and langfuse_configured():
    from langfuse.langchain import CallbackHandler

    live_app = build_agent(
        ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=512)
    )
    live_app.invoke(
        {"messages": [HumanMessage(CHAIN_TASK)], "step": 0},
        config={"callbacks": [CallbackHandler()]},
    )
    print("\nLive run sent to Langfuse -- open the L11 project to see the spans.")
else:
    print("\n(Set ANTHROPIC_API_KEY + Langfuse to send the SAME run to the L11 dashboard.)")

inline trace (graph run):
  HumanMessage Use the calculator to compute 21 * 2, then tell 
  AIMessage    -> calculator
  ToolMessage  42
  AIMessage    -> lookup
  ToolMessage  37000000
  AIMessage    21*2 = 42, and Tokyo's population is 37000000.

(Set ANTHROPIC_API_KEY + Langfuse to send the SAME run to the L11 dashboard.)


[↑ Back to top](#top)

## 5. Takeaways

- **Bring your eval set across.** The L12 `EvalCase`s and scorers ran **unchanged** against the
  LangGraph rebuild -- one thin adapter mapped the graph's output into a `RunResult`. Same cases
  pass -> the rebuild is correct.
- **The adapter is the whole trick.** Evaluation travels with the *agent's behavior* (its tool
  path, its answer), not with the implementation -- so a `messages`-dict and a `RunResult` are
  interchangeable to a scorer.
- **Against a live model, read the pass *rate*, not a single run.** A couple of flaky cases is
  L12's non-determinism lesson, not a port bug -- bump `samples=` and read the rate.
- **Tracing transfers, too.** The graph's run lands in the same L11 Langfuse, as the same
  GENERATION/SPAN spans -- you didn't relearn observation, you reused it.
- **You've now rebuilt, evaluated, and traced the agent as a graph.** Next: L15 surveys the
  *patterns* this graph shape composes into ([L1408](L1408_lecture.md)).

[↑ Back to top](#top)